# Colab + GitHub pipeline

Notebook base para trabajar el proyecto desde Colab Pro+ conectado a GitHub.

Este notebook está pensado para:
- clonar el repositorio
- montar Google Drive
- cargar datos
- reconstruir clusters
- construir adyacencias
- dejar listo el baseline GCN + LSTM


In [ ]:
# 1. Clonar el repositorio
REPO_URL = 'https://github.com/TU_USUARIO/traffic_gnn_madrid.git'
!rm -rf /content/traffic_gnn_madrid
!git clone {REPO_URL}
%cd /content/traffic_gnn_madrid


In [ ]:
# 2. Instalar dependencias
!pip install -q -r requirements.txt


In [ ]:
# 3. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 4. Añadir src al path
import sys
from pathlib import Path
sys.path.append(str(Path('/content/traffic_gnn_madrid/src')))


In [ ]:
# 5. Imports principales
import pandas as pd
import numpy as np
import torch

from torch.utils.data import DataLoader

from traffic_gnn.data.io import read_traffic_csv, read_sensors_csv
from traffic_gnn.features.congestion import calcular_congestion
from traffic_gnn.features.temporal import obtener_variables_temporales
from traffic_gnn.clustering.behavior import calcular_pivote_cl_comp, generar_cluster_comportamiento
from traffic_gnn.clustering.proximity import generar_cluster_proximidad
from traffic_gnn.clustering.intersections import intersectar_clusters
from traffic_gnn.graph.aggregation import aggregation_congestion_por_clusters, calcular_centroides_clusters
from traffic_gnn.graph.adjacency import adjacency_knn_from_centroids, build_cluster_time_matrix, adjacency_correlation_topk, to_pyg_tensors
from traffic_gnn.graph.datasets import crear_ventanas, split_temporal, build_tensor_datasets
from traffic_gnn.models.gcn_lstm import GCN_LSTM
from traffic_gnn.training.engine import train_one_epoch, evaluate


## 6. Configura las rutas de tus datos

Ajusta estas rutas según tu Drive.

In [ ]:
TRAFFIC_PATH = '/content/drive/MyDrive/TU_RUTA/01-2025.csv'
SENSORS_PATH = '/content/drive/MyDrive/TU_RUTA/pmed_ubicacion_01-2025.csv'


In [ ]:
cols_trafico = ['id', 'fecha', 'tipo_elem', 'intensidad', 'ocupacion']
cols_puntos = ['id', 'distrito', 'nombre', 'utm_x', 'utm_y', 'longitud', 'latitud']

trafico_raw = read_traffic_csv(TRAFFIC_PATH, usecols=cols_trafico)
sensores_raw = read_sensors_csv(SENSORS_PATH, usecols=cols_puntos)

print(trafico_raw.shape)
print(sensores_raw.shape)


## 7. Preparación base

In [ ]:
trafico = calcular_congestion(trafico_raw)
trafico = obtener_variables_temporales(trafico)

pivote_cl_comp = calcular_pivote_cl_comp(trafico)
pivote_cl_comp = pivote_cl_comp.dropna()
lista_ids_validos = pivote_cl_comp.index

sensores = sensores_raw[sensores_raw['id'].isin(lista_ids_validos)].copy()
trafico = trafico[trafico['id'].isin(lista_ids_validos)].copy()

print('Sensores válidos:', sensores['id'].nunique())


## 8. Caso: proximidad

In [ ]:
EPS_PROX_500 = 171.63
cluster_proximidad_500 = generar_cluster_proximidad(sensores, epsilon=EPS_PROX_500)
cluster_proximidad_500 = cluster_proximidad_500[['id', 'nombre', 'cluster_proximidad']]
cluster_proximidad_500.head()


## 9. Caso: proximidad ∩ comportamiento

In [ ]:
pivote_clusterizado = generar_cluster_comportamiento(pivote_cl_comp, n_clusters=2)
cluster_comportamiento = pivote_clusterizado.reset_index()[['id', 'cluster_comportamiento']]

cluster_prox_tmp = generar_cluster_proximidad(sensores, epsilon=195.43)[['id', 'nombre', 'cluster_proximidad']]
cluster_prox_comp_500 = intersectar_clusters(cluster_prox_tmp, cluster_comportamiento)
cluster_prox_comp_500.head()


## 10. Agregación por cluster

In [ ]:
df_merge = trafico.merge(cluster_proximidad_500[['id', 'cluster_proximidad']], on='id', how='left')
agg = aggregation_congestion_por_clusters(df_merge, nombre_col_cluster='cluster_proximidad')
agg.head()


## 11. Adyacencia por cercanía

In [ ]:
info_centroides = cluster_proximidad_500.merge(sensores[['id', 'utm_x', 'utm_y']], on='id', how='left')
centroides = calcular_centroides_clusters(info_centroides, cluster_col='cluster_proximidad')
edges_knn = adjacency_knn_from_centroids(centroides, k=5)
edges_knn.head()


## 12. Adyacencia por correlación

In [ ]:
tabla = build_cluster_time_matrix(agg)
tabla_gnn, edges_corr, mapping, mapping_inverso = adjacency_correlation_topk(tabla, k=5)
edge_index, edge_weight = to_pyg_tensors(edges_corr)

print(tabla_gnn.shape)
print(edge_index.shape)
print(edge_weight.shape)


## 13. Ventanas temporales

In [ ]:
data = tabla_gnn.values.astype(np.float32)
X, y = crear_ventanas(data, window=12, horizon=1)
X_train, y_train, X_val, y_val, X_test, y_test = split_temporal(X, y)

train_dataset, val_dataset, test_dataset = build_tensor_datasets(X_train, y_train, X_val, y_val, X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(X_train.shape, y_train.shape)


## 14. Modelo GCN + LSTM

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN_LSTM(num_nodes=X_train.shape[2], in_channels=1, gcn_hidden=32, lstm_hidden=64).to(device)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
edge_index = edge_index.to(device)
edge_weight = edge_weight.to(device)
print(device)


## 15. Entrenamiento rápido de prueba

In [ ]:
epochs = 3
best_val_loss = float('inf')
best_state = None

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, edge_index, edge_weight, device)
    val_loss, val_mae, val_rmse, _, _ = evaluate(model, val_loader, criterion, edge_index, edge_weight, device)
    print(f'Epoch {epoch+1:02d} | Train {train_loss:.6f} | Val {val_loss:.6f} | MAE {val_mae:.6f} | RMSE {val_rmse:.6f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = model.state_dict()


## 16. Evaluación

In [ ]:
model.load_state_dict(best_state)
test_loss, test_mae, test_rmse, y_pred_test, y_true_test = evaluate(model, test_loader, criterion, edge_index, edge_weight, device)
print('Test loss:', test_loss)
print('Test MAE :', test_mae)
print('Test RMSE:', test_rmse)
